In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from qdrant_client import QdrantClient, models
import sys
import asyncio
import nest_asyncio

sys.path.append("/Users/tejanshu/Desktop/Projects/llm-project/LegalMind/LegalMind/notebooks")
from search import hybrid_search

# ------------------ Setup ------------------
nest_asyncio.apply()
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.llm7.io/v1"
)

sim_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
qd_client = QdrantClient("http://localhost:6333")

# ------------------ Load Questions ------------------
df = pd.read_csv("../data/all-questions.csv")
sampled_questions = pd.concat([
    df[df.collection == "collection-new"].sample(frac=0.2, random_state=42),
    df[df.collection == "collection-old"].sample(frac=0.2, random_state=42),
    df[df.collection == "collection-map"].sample(frac=0.2, random_state=42)
]).reset_index(drop=True)

# ------------------ Pick 200 Questions ------------------
best_questions = sampled_questions.sample(n=200, random_state=42).reset_index(drop=True)

# ------------------ Prompts ------------------
prompts = {
    "Prompt 1":
    """You are a legal assistant AI for Indian Criminal laws.

Rules:

1. Normal questions:
   - Crime: Define the act using BNS.
   - Procedure: Explain how authorities act using BNSS.
   - Evidence: Explain what evidence can be used using BSA.
   - Mention the relevant law (BNS, BNSS, BSA, IPC, CrPC, IEA) wherever applicable.
   - Keep language simple and understandable by ordinary people.

2. Direct section/chapter questions:
   - Explain the section directly and clearly using the relevant law.
   - Skip the usual crime-procedure-evidence format.
   - EXPLAIN the exact section that the user asked you.

3. Showing changes between old and new laws:
   - Use collection-map.
   - Explain with collection-new and collection-old: show old section, new section, subject, and what changed.

4. Restrictions:
   - Do NOT hallucinate.
   - If context is missing, reply: "Sorry, can you rephrase the question?".
   - Only use IPC, CrPC, IEA if explicitly asked.
   - Do NOT use your knowledge to say something.
   - Do NOT tell anything about context in the answer.

Context:
{context}

Question: {question}

Answer:""",


"Prompt 2": 
    """You are a law expert AI specialized in BNS, BNSS, BSA, and can refer to IPC, CrPC, IEA if explicitly asked.

1. For general queries:
   - Step 1: **Crime** – what the act is using **BNS**.
   - Step 2: **Procedure** – how authorities act using **BNSS**.
   - Step 3: **Evidence** – what evidence can be used using **BSA**.
   - **Mention the relevant law** wherever applicable.
   - Keep explanations simple and citizen-friendly.

2. For direct section or chapter queries:
   - Provide a **direct, clear explanation** of the section using the relevant law.
   - Skip the stepwise format.

3. For showing changes between old and new laws:
   - Use **collection-map** and explain using **collection-new** and **collection-old**: old section, new section, subject, summary of changes.

4. Restrictions:
   - Do NOT hallucinate; if context is missing, reply: `"Sorry, can you rephrase the question?"`.
   - Only refer to IPC, CrPC, IEA if explicitly asked.

Context:
{context}

Question: {question}

Answer:""",



    "Prompt 3": 
    """You are a legal assistant AI that explains Indian laws (BNS, BNSS, BSA) in simple terms for ordinary citizens.

1. Normal questions:
   - **Crime:** What the act is (BNS).
   - **Procedure:** How authorities handle it (BNSS).
   - **Evidence:** What evidence is acceptable in court (BSA).
   - **Mention the relevant law** wherever applicable.
   - Keep explanations simple and easy to understand.

2. Direct section, chapter, or provision questions:
   - Explain the section **directly and clearly** using the relevant law.
   - Skip the usual crime-procedure-evidence format.

3. Showing changes between old and new laws:
   - Use **collection-map**.
   - Explain using **collection-new** and **collection-old**: show old section, new section, subject, and summary of changes.

4. Do not:
   - Hallucinate; if no information is available, respond: `"Sorry, can you rephrase the question?"`.
   - Use IPC, CrPC, IEA unless explicitly asked.



Context:
{context}

Question: {question}

Answer:"""
}

# ------------------ Generate RAG Answer (sync) ------------------
def generate_rag_answer_sync(question: str, prompt_template: str, alpha=0.3):
    results = hybrid_search(question, alpha=alpha)
    context = ""
    for collection, docs in results.items():
        for d in docs:
            text = d.get("text", "")
            fields = d.get("fields") or d.get("metadata") or {}
            fields_str = ", ".join([f"{k}: {v}" for k, v in fields.items()])
            context += f"{text}\n[{fields_str}]\n"

    if not context.strip():
        return ""

    prompt = prompt_template.format(context=context, question=question)
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )
        try:
            return response.choices[0].message.content.strip()
        except AttributeError:
            return response.choices[0].content.strip()
    except Exception as e:
        print(f"⚠️ LLM call failed for question: {question}\nError: {e}")
        return ""

# ------------------ Async wrapper ------------------
async def generate_rag_answer_async(question: str, prompt_template: str):
    return await asyncio.to_thread(generate_rag_answer_sync, question, prompt_template)

# ------------------ Get Actual Chunks ------------------
def get_actual_chunks(q_id, collection_name):
    hits, _ = qd_client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[models.FieldCondition(key="chunk_id", match=models.MatchValue(value=q_id))]
        ),
        limit=1
    )
    if not hits:
        return ""
    texts = []
    for p in hits:
        payload = p.payload or {}
        if collection_name == "collection-map":
            fields = payload.get("fields", {})
            texts.append(fields.get("Summary_of_comparison", ""))
        else:
            texts.append(payload.get("text") or payload.get("content", ""))
    return " ".join(texts)

# ------------------ Cosine Similarity ------------------
def cosine_sim(text1, text2):
    if not text1.strip() or not text2.strip():
        return 0
    emb1 = sim_model.encode([text1])[0]
    emb2 = sim_model.encode([text2])[0]
    return cosine_similarity([emb1], [emb2])[0][0]

# ------------------ Evaluate Single Question ------------------
async def evaluate_question_async(q):
    q_id = q["q_id"]
    question_text = q["question"]
    collection_name = q["collection"]

    actual_text = get_actual_chunks(q_id, collection_name)
    if not actual_text.strip():
        return None

    # Run three prompts concurrently
    tasks = [generate_rag_answer_async(question_text, prompt) for prompt in prompts.values()]
    ans1, ans2, ans3 = await asyncio.gather(*tasks)

    sim1 = cosine_sim(ans1, actual_text)
    sim2 = cosine_sim(ans2, actual_text)
    sim3 = cosine_sim(ans3, actual_text)

    return {
        "q_id": q_id,
        "collection": collection_name,
        "question": question_text,
        "answer_prompt1": ans1,
        "answer_prompt2": ans2,
        "answer_prompt3": ans3,
        "actual_text": actual_text,
        "similarity_p1": sim1,
        "similarity_p2": sim2,
        "similarity_p3": sim3
    }

# ------------------ Main Async Execution ------------------
async def main():
    results_list = []
    sem = asyncio.Semaphore(10)  # limit concurrent LLM calls

    async def sem_evaluate(q):
        async with sem:
            return await evaluate_question_async(q)

    tasks = [sem_evaluate(q) for _, q in best_questions.iterrows()]
    for future in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        result = await future
        if result:
            results_list.append(result)

    # Compute averages
    scores = {"Prompt 1": [], "Prompt 2": [], "Prompt 3": []}
    for r in results_list:
        scores["Prompt 1"].append(r["similarity_p1"])
        scores["Prompt 2"].append(r["similarity_p2"])
        scores["Prompt 3"].append(r["similarity_p3"])

    avg_scores = {k: (sum(v) / len(v) if v else 0) for k, v in scores.items()}
    best_prompt = max(avg_scores, key=avg_scores.get)

    print("\n--- Evaluation Results ---")
    for k, v in avg_scores.items():
        print(f"{k}: {v:.4f}")
    print(f"\n✅ Best Overall Prompt: {best_prompt}")

    pd.DataFrame(results_list).to_csv("prompt_comparison_results_200.csv", index=False)
    print("\nResults saved to prompt_comparison_results_200.csv")

# ------------------ Run in Notebook ------------------
await main()


 10%|█         | 20/200 [08:47<51:18, 17.10s/it]   

⚠️ LLM call failed for question: Which act and section are cited as adding Section 5 to the IPC in this excerpt?
Error: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>llm7.io | 524: A timeout occurred</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
  

 12%|█▏        | 23/200 [09:28<49:34, 16.80s/it]

⚠️ LLM call failed for question: As per the Note in the summons, what is the maximum fine allowed?
Error: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>llm7.io | 524: A timeout occurred</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 

 81%|████████  | 162/200 [28:47<06:36, 10.43s/it] 

⚠️ LLM call failed for question: What is the official section title for CrPC 479?
Error: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>llm7.io | 524: A timeout occurred</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-blo

 89%|████████▉ | 178/200 [33:23<03:26,  9.40s/it]

⚠️ LLM call failed for question: Is an illegal omission within the scope of 'words referring to acts' as described in IPC Section 32?
Error: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>llm7.io | 524: A timeout occurred</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-24

100%|██████████| 200/200 [39:36<00:00, 11.88s/it]


--- Evaluation Results ---
Prompt 1: 0.6895
Prompt 2: 0.6883
Prompt 3: 0.6699

✅ Best Overall Prompt: Prompt 1

Results saved to prompt_comparison_results_200.csv
